# ArgLLM baseline (DF-QuAD) — DeepSeek-V3 on CosmosQA (500 claims, earlier sample from the training split)

This is the **earlier adaptation** of the ArgLLM prompts described in Section 5.1 of the paper: the model writes its own facts, and the argument prompt asks for one argument on each side instead of allowing the model to decline. Strengths are computed with the authors' DF-QuAD implementation (Uncertainpy). In `dfquad_pred`, the second field of the claim `Argument` is only its text label; DF-QuAD uses the base strength, never the gold label.

Run on Kaggle in batches of 100; the API key is read from Kaggle secrets (`DEEPSEEK_API_KEY`).

In [ ]:
!pip install -q openai datasets scikit-learn
import os, sys, glob, re, time, pickle, random
from pathlib import Path
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from datasets import load_dataset

WORK_DIR = Path("/kaggle/working/cosmos_dfquad"); WORK_DIR.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=UserSecretsClient().get_secret("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")
MODEL_NAME = "deepseek-chat"

# SAME CosmosQA claims (random_state=42, identical to extension file)
base = "https://huggingface.co/datasets/allenai/cosmos_qa/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"
df = pd.read_parquet(base)
random.seed(42)
rows = []
sample = df.sample(400, random_state=42).reset_index(drop=True)
for _, r in sample.iterrows():
    ctx = str(r["context"]).strip(); q = str(r["question"]).strip()
    ans = {"0": r["answer0"], "1": r["answer1"], "2": r["answer2"], "3": r["answer3"]}
    ck = str(r["label"]).strip()
    if ck not in ans: continue
    correct = str(ans[ck]).strip()
    wrong_keys = [k for k in ans if k != ck and "none of the above" not in str(ans[k]).lower()]
    if not wrong_keys: continue
    wrong = str(ans[random.choice(wrong_keys)]).strip()
    rows.append({"claim": f"{ctx} {q} {correct}", "gold": 1})
    rows.append({"claim": f"{ctx} {q} {wrong}", "gold": 0})
cdf = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
t = cdf[cdf["gold"]==1].head(250); f = cdf[cdf["gold"]==0].head(250)
baf_df = pd.concat([t,f]).sample(frac=1, random_state=42).reset_index(drop=True)
baf_df["example_index"] = range(len(baf_df))
print("CosmosQA claims:", len(baf_df), "| first:", baf_df.iloc[0]["claim"][:70])

def generate_facts(claim):
    prompt = f"""You are building atomic factual building blocks.
For the claim below, produce 3-5 SHORT, SEPARATE factual statements. Each must be:
- A single atomic fact about ONE concept in the claim.
- General commonsense knowledge, stated on its own.
- NOT a comparison, ranking, or conclusion. NOT about whether the claim is true or false.
Claim: "{claim}"
Return 3-5 atomic facts, one per line. Do not conclude."""
    for _ in range(5):
        try:
            resp=client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"system","content":"You output atomic facts. You never conclude."},
                          {"role":"user","content":prompt}], temperature=0.3, max_tokens=400)
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(8); continue
            return ""
    return ""
print("✅ CELL 1 ready")

CosmosQA claims: 500 | first: It made me think of my mom and miss her a lot ... she loves cactus gar
✅ CELL 1 ready


In [ ]:
if not glob.glob("/kaggle/working/**/Uncertainpy/src", recursive=True):
    os.system("cd /kaggle/working && git clone -q https://github.com/CLArg-group/argumentative-llms.git")
UNC = glob.glob("/kaggle/working/**/Uncertainpy/src", recursive=True)[0]
sys.path.insert(0, UNC)
from uncertainpy.gradual import BAG, Argument
from uncertainpy.gradual.algorithms.Acyclic import computeStrengthValues
from uncertainpy.gradual.semantics.modular import ProductAggregation, LinearInfluence
agg_f = ProductAggregation(); inf_f = LinearInfluence(conservativeness=1)

def llm_raw(prompt, mt=200):
    for _ in range(5):
        try:
            r=client.chat.completions.create(model=MODEL_NAME,messages=[{"role":"user","content":prompt}],temperature=0.7,max_tokens=mt,top_p=0.95)
            return r.choices[0].message.content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(8); continue
            time.sleep(3)
    return ""
def am_prompt(claim, facts, support):
    verb="supporting" if support else "attacking"; rel="support" if support else "attack"
    return (f'Please provide a single short argument {verb} the following claim, using the facts as context. '
            f'Construct the argument so it refers to the truthfulness of the claim. '
            f'You must always produce a {rel} argument, even if weak; do not return N/A.\n'
            f'Facts:\n{facts}\n\nClaim: {claim}\nArgument:')
def ue_arg_prompt(argument, claim, support):
    favour="in favour of" if support else "against"; rel="supports" if support else "refutes"
    return (f'You are an analyst evaluating arguments. For the argument:\n\n"{argument}"\n\n'
            f'give your confidence it makes a compelling case {favour}:\n\n"{claim}"\n\n'
            f'Base it on how well the argument {rel} the statement and its accuracy. Respond 0-100%. Format:\nLikelihood:')
def ue_topic_prompt(claim, facts):
    return (f'You are an analyst. Using these facts as context:\n{facts}\n\nFor the statement:\n\n"{claim}"\n\n'
            f'give your confidence it is correct and truthful, 0-100%. Format:\nLikelihood:')
def gen_argument(claim, facts, support):
    for _ in range(2):
        txt=llm_raw(am_prompt(claim,facts,support),150); txt=re.sub(r'\*+','',txt); txt=re.sub(r'^\s*Argument\s*:\s*','',txt,flags=re.I).strip()
        lines=[l.strip() for l in txt.split("\n") if l.strip()]; txt=lines[0] if lines else txt
        if txt and "N/A" not in txt: return txt.strip('"').strip()
    return "N/A"
def get_lik(prompt):
    txt=llm_raw(prompt,150); m=re.search(r'Likelihood:\s*([0-9]{1,3})\s*%',txt) or re.search(r'([0-9]{1,3})\s*%',txt)
    return max(0,min(1,int(m.group(1))/100)) if m else 0.5
def arg_base(a,c,s): return None if a=="N/A" else get_lik(ue_arg_prompt(a,c,s))
def topic_base(c,f): return get_lik(ue_topic_prompt(c,f))
print("✅ CELL 2 ready — DF-QuAD loaded")

✅ CELL 2 ready — DF-QuAD loaded


In [ ]:
DFQ_CKPT = WORK_DIR / "cosmos_dfquad_ckpt.pkl"
def load_dfq(): return pickle.load(open(DFQ_CKPT,"rb")) if DFQ_CKPT.exists() else {}
def save_dfq(d): pickle.dump(d, open(DFQ_CKPT,"wb"))
def run_dfquad_batch(n=100):
    done = load_dfq()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)} (DF-QuAD)...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx]; claim = row["claim"]
        try:
            facts = generate_facts(claim)
            sup = gen_argument(claim, facts, True); att = gen_argument(claim, facts, False)
            res = {"gold": int(row["gold"]), "claim": claim, "facts": facts, "sup": sup, "att": att,
                   "sup_base": arg_base(sup, claim, True), "att_base": arg_base(att, claim, False),
                   "topic_base": topic_base(claim, facts)}
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_dfq(); d[idx] = res; save_dfq(d)
        if k % 10 == 0 or k == len(todo): print(f"  {k}/{len(todo)} → {len(load_dfq())} total")
    print(f"✅ {len(load_dfq())}/{len(baf_df)} done")
run_dfquad_batch(100)

Have 0/500. Running 100 (DF-QuAD)...
  10/100 → 10 total
  20/100 → 20 total
  30/100 → 30 total
  40/100 → 40 total
  50/100 → 50 total
  60/100 → 60 total
  70/100 → 70 total
  80/100 → 80 total
  90/100 → 90 total
  100/100 → 100 total
✅ 100/500 done


In [ ]:
def dfquad_pred(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

dfq = load_dfq(); idxs = sorted(dfq); gold = [dfq[i]["gold"] for i in idxs]; n=len(idxs)
print(f"=== CosmosQA DF-QuAD ({n}) ===")
for est, lab in [(False,"0.5 Base (fair)"), (True,"Est. Base")]:
    pred = [int(dfquad_pred(dfq[i], est) > 0.5) for i in idxs]
    acc = accuracy_score(gold, pred)*100; f1 = f1_score(gold, pred, zero_division=0)*100
    tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {acc:.1f}%  (F1 {f1:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"\n  >>> Extension: 67% skep / 64% cred | net_support: 60% <<<")

=== CosmosQA DF-QuAD (100) ===
  DF-QuAD 0.5 Base (fair): 62.0%  (F1 52.5)  TN=41 FP=13 FN=25 TP=21
  DF-QuAD Est. Base: 79.0%  (F1 78.8)  TN=40 FP=14 FN=7 TP=39

  >>> Extension: 67% skep / 64% cred | net_support: 60% <<<


In [ ]:
# Run to 200
run_dfquad_batch(100)

# Back up (green button)
import base64, zipfile
from IPython.display import HTML, display
data=[load_dfq()[k] for k in sorted(load_dfq())]
N=len(data); pkl=WORK_DIR/f"cosmos_dfquad_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"cosmos_dfquad_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"cosmos_dfquad_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="cosmos_dfquad_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download cosmos_dfquad_{N}_backup.zip</a>'))
print(f"Backed up {N}\n")

# Eval
def dfquad_pred(v, est):
    bag=BAG(); tau=v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim=Argument("db0",v["gold"],float(tau)); bag.arguments["db0"]=claim
    if v["sup"]!="N/A" and v["sup_base"] is not None: bag.add_support(Argument("S1","s",float(v["sup_base"])),claim)
    if v["att"]!="N/A" and v["att_base"] is not None: bag.add_attack(Argument("A1","a",float(v["att_base"])),claim)
    computeStrengthValues(bag,agg_f,inf_f); return float(bag.arguments["db0"].strength)
dfq=load_dfq(); idxs=sorted(dfq); gold=[dfq[i]["gold"] for i in idxs]; n=len(idxs)
print(f"=== CosmosQA DF-QuAD ({n}) ===")
for est,lab in [(False,"0.5 Base (fair)"),(True,"Est. Base")]:
    pred=[int(dfquad_pred(dfq[i],est)>0.5) for i in idxs]
    tn,fp,fn,tp=confusion_matrix(gold,pred,labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {accuracy_score(gold,pred)*100:.1f}%  (F1 {f1_score(gold,pred,zero_division=0)*100:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")

Have 100/500. Running 100 (DF-QuAD)...
  10/100 → 110 total
  20/100 → 120 total
  30/100 → 130 total
  40/100 → 140 total
  50/100 → 150 total
  60/100 → 160 total
  70/100 → 170 total
  80/100 → 180 total
  90/100 → 190 total
  100/100 → 200 total
✅ 200/500 done


Backed up 200

=== CosmosQA DF-QuAD (200) ===
  DF-QuAD 0.5 Base (fair): 60.0%  (F1 55.1)  TN=71 FP=24 FN=56 TP=49
  DF-QuAD Est. Base: 75.5%  (F1 79.0)  TN=59 FP=36 FN=13 TP=92


The Kaggle session was restarted after 200 claims; the next cells reload the saved 200 results and continue.

In [ ]:
import glob
hits = glob.glob("/kaggle/input/**/cosmos_dfquad_200.pkl", recursive=True)
print("found:", hits)
data = pickle.load(open(hits[0], "rb"))
print("type:", type(data), "| len:", len(data))
print("first item keys:", list(data[0].keys()) if isinstance(data, list) else list(data.keys())[:3])

found: ['/kaggle/input/datasets/nematchokri/cosmos-dfquad-200/cosmos_dfquad_200.pkl']
type: <class 'list'> | len: 200
first item keys: ['gold', 'claim', 'facts', 'sup', 'att', 'sup_base', 'att_base', 'topic_base']


In [ ]:
DFQ_CKPT = WORK_DIR / "cosmos_dfquad_ckpt.pkl"
def load_dfq(): return pickle.load(open(DFQ_CKPT,"rb")) if DFQ_CKPT.exists() else {}
def save_dfq(d): pickle.dump(d, open(DFQ_CKPT,"wb"))
def run_dfquad_batch(n=100):
    done = load_dfq()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)} (DF-QuAD)...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx]; claim = row["claim"]
        try:
            facts = generate_facts(claim)
            sup = gen_argument(claim, facts, True); att = gen_argument(claim, facts, False)
            res = {"gold": int(row["gold"]), "claim": claim, "facts": facts, "sup": sup, "att": att,
                   "sup_base": arg_base(sup, claim, True), "att_base": arg_base(att, claim, False),
                   "topic_base": topic_base(claim, facts)}
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_dfq(); d[idx] = res; save_dfq(d)
        if k % 10 == 0 or k == len(todo): print(f"  {k}/{len(todo)} → {len(load_dfq())} total")
    print(f"✅ {len(load_dfq())}/{len(baf_df)} done")
print("✅ runner ready")

✅ runner ready


In [ ]:
path = "/kaggle/input/datasets/nematchokri/cosmos-dfquad-200/cosmos_dfquad_200.pkl"
data = pickle.load(open(path, "rb"))
restored = {}
unmatched = 0
for r in data:
    idx = baf_df.index[baf_df["claim"] == r["claim"]]
    if len(idx): restored[int(idx[0])] = r
    else: unmatched += 1
save_dfq(restored)
print(f"Restored {len(load_dfq())} examples | unmatched: {unmatched}")

Restored 200 examples | unmatched: 0


In [ ]:
run_dfquad_batch(100)   # → 300

Have 200/500. Running 100 (DF-QuAD)...
  10/100 → 210 total
  20/100 → 220 total
  30/100 → 230 total
  40/100 → 240 total
  50/100 → 250 total
  60/100 → 260 total
  70/100 → 270 total
  80/100 → 280 total
  90/100 → 290 total
  100/100 → 300 total
✅ 300/500 done


In [ ]:
def dfquad_pred(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

dfq = load_dfq(); idxs = sorted(dfq); gold = [dfq[i]["gold"] for i in idxs]; n=len(idxs)
print(f"=== DeepSeek CosmosQA DF-QuAD ({n}) ===")
for est, lab in [(False,"0.5 Base (fair)"), (True,"Est. Base")]:
    pred = [int(dfquad_pred(dfq[i], est) > 0.5) for i in idxs]
    acc = accuracy_score(gold, pred)*100; f1 = f1_score(gold, pred, zero_division=0)*100
    tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {acc:.1f}%  (F1 {f1:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"\n  >>> Extension: ~62-67% | net_support: ~61% <<<")

=== DeepSeek CosmosQA DF-QuAD (300) ===
  DF-QuAD 0.5 Base (fair): 59.7%  (F1 52.5)  TN=112 FP=38 FN=83 TP=67
  DF-QuAD Est. Base: 74.7%  (F1 78.2)  TN=88 FP=62 FN=14 TP=136

  >>> Extension: ~62-67% | net_support: ~61% <<<


In [ ]:
run_dfquad_batch(100)   

Have 300/500. Running 100 (DF-QuAD)...
  10/100 → 310 total
  20/100 → 320 total
  30/100 → 330 total
  40/100 → 340 total
  50/100 → 350 total
  60/100 → 360 total
  70/100 → 370 total
  80/100 → 380 total
  90/100 → 390 total
  100/100 → 400 total
✅ 400/500 done


In [ ]:
def dfquad_pred(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

dfq = load_dfq(); idxs = sorted(dfq); gold = [dfq[i]["gold"] for i in idxs]; n=len(idxs)
print(f"=== DeepSeek CosmosQA DF-QuAD ({n}) ===")
for est, lab in [(False,"0.5 Base (fair)"), (True,"Est. Base")]:
    pred = [int(dfquad_pred(dfq[i], est) > 0.5) for i in idxs]
    acc = accuracy_score(gold, pred)*100; f1 = f1_score(gold, pred, zero_division=0)*100
    tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {acc:.1f}%  (F1 {f1:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"\n  >>> Extension: ~62-67% | net_support: ~61% <<<")

=== DeepSeek CosmosQA DF-QuAD (400) ===
  DF-QuAD 0.5 Base (fair): 57.0%  (F1 50.0)  TN=142 FP=53 FN=119 TP=86
  DF-QuAD Est. Base: 75.0%  (F1 79.0)  TN=112 FP=83 FN=17 TP=188

  >>> Extension: ~62-67% | net_support: ~61% <<<


In [ ]:
run_dfquad_batch(100)   

Have 400/500. Running 100 (DF-QuAD)...
  10/100 → 410 total
  20/100 → 420 total
  30/100 → 430 total
  40/100 → 440 total
  50/100 → 450 total
  60/100 → 460 total
  70/100 → 470 total
  80/100 → 480 total
  90/100 → 490 total
  100/100 → 500 total
✅ 500/500 done


Final results on 500 claims. The last four printed lines (our approach, net support) are copied from the extension notebook for comparison; they are not computed here.

In [ ]:
def dfquad_pred(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

dfq = load_dfq(); idxs = sorted(dfq); gold = [dfq[i]["gold"] for i in idxs]; n=len(idxs)
print(f"=== DeepSeek CosmosQA DF-QuAD FINAL ({n}) ===")
for est, lab in [(False,"0.5 Base (fair)"), (True,"Est. Base")]:
    pred = [int(dfquad_pred(dfq[i], est) > 0.5) for i in idxs]
    acc = accuracy_score(gold, pred)*100; f1 = f1_score(gold, pred, zero_division=0)*100
    tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {acc:.1f}%  (F1 {f1:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"\n=== DeepSeek CosmosQA COMPLETE (500) ===")
print(f"  Extension skeptical: 66.6%  ← best")
print(f"  Extension credulous: 64.0%")
print(f"  net_support:         58.0%")
print(f"  DF-QuAD 0.5 Base:    (above)  ← fair comparison")

=== DeepSeek CosmosQA DF-QuAD FINAL (500) ===
  DF-QuAD 0.5 Base (fair): 58.4%  (F1 51.2)  TN=183 FP=67 FN=141 TP=109
  DF-QuAD Est. Base: 72.8%  (F1 76.9)  TN=137 FP=113 FN=23 TP=227

=== DeepSeek CosmosQA COMPLETE (500) ===
  Extension skeptical: 66.6%  ← best
  Extension credulous: 64.0%
  net_support:         58.0%
  DF-QuAD 0.5 Base:    (above)  ← fair comparison
